# UCS420 Cognitive Computing - Lab Assignment 9
## Landing AI: Vision-Based Cognitive Systems

**Project 1:** Cognitive assistant for reading medicine-bottle labels (object detection + OCR)  
**Project 2:** Emotion-aware cognitive assistant using facial-expression classification  

**Student name:** Mehakdeep Kaur  
**Roll number:** 1024170364

This notebook is designed for Jupyter Notebook or Google Colab. It combines (1) LandingLens platform steps, (2) optional live inference through your own LandingAI endpoint credentials, and (3) analysis of the result exports supplied with the assignment.

> Important: the CSV exports are evaluation records, not the original image datasets. They let us analyze your trained models, but they cannot independently retrain the models because the actual image pixels are not included in those CSV files.

## How to run this notebook

1. Download and extract the accompanying ZIP so that `Q1CSV.csv`, `Q2 CSV.csv`, and the screenshots are in the same folder as this notebook.
2. Run the installation and import cells.
3. Run the medicine-label result analysis and emotion-model evaluation cells. These use the supplied CSVs and do not need an API key.
4. The two live inference sections are optional. Run them only after training and deploying your own LandingLens models and obtaining an API key and endpoint ID. Never paste an API key into a public notebook or share it in screenshots.

The provided outputs show a **single-class medicine-label detector** and a **five-class emotion classifier**. The emotion categories found in the CSV are `neutral`, `happy`, `angry`, `surprise`, and `sad`.

## 1. Install libraries

Run this cell once in a fresh environment. `landingai` is used for calling a deployed LandingLens model and its OCR service. `opencv-python-headless` is used for image cropping and face detection in notebook environments.

In [ ]:
%pip install -q landingai opencv-python-headless pandas numpy matplotlib seaborn scikit-learn pillow

## 2. Import Python libraries

In [ ]:
from pathlib import Path
import json
import re
import getpass
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from IPython.display import display, Markdown

print("Libraries imported successfully.")

# PROJECT 1 - Medicine Bottle Label Reading Assistant

## 3. Aim and system design

The assistant accepts a photo of a medicine bottle, detects the label region, crops it, extracts printed words using OCR, and highlights likely fields such as medicine name, strength, batch number, manufacturing date, expiry date, and MRP. It should display the OCR confidence and ask the user to verify the result.

**Pipeline:**

`Bottle photo -> LandingLens object detector -> medicine-label bounding box -> crop -> OCR -> field extraction -> readable summary`

LandingLens distinguishes object detection (find and locate an object using boxes) from classification (assign one label to the whole image). Object detection is the appropriate project type for locating a medicine-label region. See the official [LandingAI project types guide](https://docs.landing.ai/) and the [LandingAI inference getting-started guide](https://landing-ai.github.io/landingai-python/inferences/getting-started/).

## 4. Create and train the medicine-label model in LandingLens

1. Open [LandingLens](https://app.landing.ai/) and create a new project.
2. Select **Object Detection**.
3. Upload clear medicine-bottle photographs. Include different bottle shapes, label colours, orientations, lighting conditions, and backgrounds.
4. Draw a bounding box around the complete printed label in each positive image. Use the same class name for every label, for example `medicine-label`.
5. Include images where no medicine label is present and mark them as images with nothing to label, where supported. This helps test false detections.
6. Separate the data into Train, Dev, and Test. Keep the Test set separate and do not train on it.
7. Click **Train**. Review the loss and validation charts, then inspect incorrect predictions and fix inaccurate labels.
8. Evaluate on the **Dev and Test** splits. Do not use the training-set score as proof of real-world accuracy.
9. Click **Deploy -> Cloud Deployment**, create an endpoint, and copy the endpoint ID. Create an API key from your LandingLens account settings.

LandingAI's official getting-started instructions follow the same sequence: upload images, label them, train, then deploy to an endpoint. The endpoint ID and API key are required for SDK inference.

### Dataset option

A public starting point is the [Medicine_Label object-detection dataset on Roboflow Universe](https://universe.roboflow.com/parichat-j/medicine_label-amj7d) (listed as CC BY 4.0 on its dataset page). Check its current terms and attribution instructions before reuse. You may also photograph medicine bottles yourself, ensuring the photos are permitted for your academic use. Do not include personal patient information.

## 5. Analyze the supplied medicine-label results (Q1CSV.csv)

The provided CSV contains one record per image and embeds the ground-truth and predicted boxes in the `GT-PRED JSON` column. The following code calculates the number of images, the total number of labelled instances, average prediction confidence, and intersection-over-union (IoU) between each exported ground-truth and predicted box.

In [ ]:
Q1_CSV_PATH = Path("Q1CSV.csv")  # Keep this CSV beside the notebook
if not Q1_CSV_PATH.exists():
    raise FileNotFoundError("Q1CSV.csv not found. Extract the supplied ZIP or set Q1_CSV_PATH to its location.")

medicine_results = pd.read_csv(Q1_CSV_PATH)
print("Images in export:", len(medicine_results))
print("Columns:", medicine_results.columns.tolist())
display(medicine_results.head())

In [ ]:
def calculate_iou(box_a, box_b):
    """Calculate intersection-over-union for [xmin, ymin, xmax, ymax] boxes."""
    x_left = max(box_a["xmin"], box_b["xmin"])
    y_top = max(box_a["ymin"], box_b["ymin"])
    x_right = min(box_a["xmax"], box_b["xmax"])
    y_bottom = min(box_a["ymax"], box_b["ymax"])

    intersection = max(0, x_right - x_left) * max(0, y_bottom - y_top)
    area_a = max(0, box_a["xmax"] - box_a["xmin"]) * max(0, box_a["ymax"] - box_a["ymin"])
    area_b = max(0, box_b["xmax"] - box_b["xmin"]) * max(0, box_b["ymax"] - box_b["ymin"])
    union = area_a + area_b - intersection
    return intersection / union if union else 0.0


detection_rows = []
for _, row in medicine_results.iterrows():
    try:
        parsed = json.loads(row["GT-PRED JSON"])
    except (TypeError, json.JSONDecodeError):
        continue

    for image_id, detections in parsed.items():
        for detection in detections:
            detection_rows.append({
                "Image_ID": image_id,
                "Ground_Truth": detection.get("gtDefect"),
                "Prediction": detection.get("predDefect"),
                "Confidence": float(detection.get("predConfidence", np.nan)),
                "IoU": calculate_iou(detection["gtBox"], detection["predBox"])
            })

detection_df = pd.DataFrame(detection_rows)
print("Total labelled/predicted instances in JSON:", len(detection_df))
print("Mean confidence:", round(detection_df["Confidence"].mean(), 4))
print("Mean box IoU:", round(detection_df["IoU"].mean(), 4))
print("Minimum confidence:", round(detection_df["Confidence"].min(), 4))
print("Minimum IoU:", round(detection_df["IoU"].min(), 4))
display(detection_df.head(10))

### Observed Q1 results from the supplied files

- Exported images: **87**.
- Total object instances in the JSON export: **91**.
- Mean confidence across exported object instances: **0.9724 (97.24%)**.
- Mean box IoU calculated from exported ground-truth/prediction pairs: **0.9439**.
- The uploaded LandingLens screenshot reports **100.0% F1, precision, and recall** at a confidence threshold of 0.61, for an evaluation set marked **Train set** (87 images, 91 instances).

**Interpretation and limitation:** the screenshot's 100% score is a training-set result, not an independent test result. The exported Q1 CSV contains positive `medicine-label` instances; the screenshot's `No label` row/column has zero counts. Therefore, these materials do not establish how well the model rejects ordinary objects or detects unseen bottles. Report the 100% figure as the score shown on the supplied training evaluation, and run a separate Test set with negative images before claiming real-world accuracy.

## 6. Optional live inference: detect the label and read its text

This cell calls your deployed LandingLens object-detection endpoint. It then crops every detected `medicine-label` region and sends the crop to LandingAI's OCR predictor. The official OCR example uses `OcrPredictor` with a `Frame`; individual OCR outputs provide text and confidence score.

**Before running:** deploy the object-detection model, collect its endpoint ID, and prepare a medicine-bottle image. The API key is requested securely and is not stored in the notebook.

In [ ]:
# OPTIONAL: run only after you have a deployed medicine-label endpoint.
from landingai.pipeline.image_source import Frame
from landingai.predict import Predictor, OcrPredictor

LANDINGAI_API_KEY = getpass.getpass("Enter your LandingAI API key (input hidden): ")
MEDICINE_ENDPOINT_ID = input("Enter the medicine-label detection endpoint ID: ").strip()
MEDICINE_IMAGE_PATH = input("Enter the path to a medicine bottle image: ").strip()

medicine_detector = Predictor(
    endpoint_id=MEDICINE_ENDPOINT_ID,
    api_key=LANDINGAI_API_KEY
)
medicine_ocr = OcrPredictor(api_key=LANDINGAI_API_KEY, language="en")

print("Endpoint objects initialized. The API key is not printed.")

In [ ]:
def extract_label_fields(ocr_text):
    """Extract possible label fields using simple patterns; always verify them visually."""
    patterns = {
        "expiry_date": r"(?i)\b(?:EXP(?:IRY)?|USE\s*BEFORE)\s*[:.-]?\s*([A-Z0-9./-]{3,15})",
        "manufacturing_date": r"(?i)\b(?:MFG|MFD|MANUFACTURED)\s*[:.-]?\s*([A-Z0-9./-]{3,15})",
        "batch_number": r"(?i)\b(?:BATCH(?:\s*NO)?|B\.NO|LOT)\s*[:.-]?\s*([A-Z0-9-]{2,20})",
        "mrp": r"(?i)\bMRP\s*[:.-]?\s*(?:RS\.?|₹)?\s*([0-9]+(?:\.[0-9]{1,2})?)",
        "strength_or_volume": r"(?i)\b([0-9]+(?:\.[0-9]+)?\s*(?:mg|mcg|g|ml|l|%))\b"
    }
    found = {}
    for field, pattern in patterns.items():
        match = re.search(pattern, ocr_text)
        found[field] = match.group(1) if match else "Not detected"
    return found


def read_medicine_bottle(image_path, confidence_threshold=0.60):
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    detection_frame = Frame.from_image(str(image_path))
    detection_frame.run_predict(medicine_detector)

    # Keep only medicine-label detections above the selected confidence threshold.
    selected = detection_frame.predictions.filter_label("medicine-label").filter_threshold(confidence_threshold)
    results = []

    for index, pred in enumerate(selected, start=1):
        x1, y1, x2, y2 = [int(v) for v in pred.bboxes]
        h, w = image.shape[:2]
        x1, x2 = max(0, x1), min(w, x2)
        y1, y2 = max(0, y1), min(h, y2)
        crop = image[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        crop_path = f"medicine_label_crop_{index}.jpg"
        cv2.imwrite(crop_path, crop)
        ocr_frame = Frame.from_image(crop_path)
        ocr_frame.run_predict(medicine_ocr)

        ocr_items = list(ocr_frame.predictions)
        text_lines = [item.text for item in ocr_items if getattr(item, "text", "").strip()]
        text_confidences = [float(item.score) for item in ocr_items if hasattr(item, "score")]
        combined_text = "\n".join(text_lines)

        results.append({
            "detection_confidence": float(pred.score),
            "ocr_text": combined_text,
            "ocr_confidences": text_confidences,
            "fields": extract_label_fields(combined_text),
            "crop_path": crop_path
        })

    if not results:
        print("No medicine label detected above the confidence threshold.")
    else:
        for number, item in enumerate(results, start=1):
            print(f"\nMEDICINE LABEL {number}")
            print("Detection confidence:", round(item["detection_confidence"], 3))
            print("OCR text:\n", item["ocr_text"])
            print("Possible fields:", item["fields"])
            print("OCR confidence values:", item["ocr_confidences"])

    print("\nSAFETY: OCR can misread medicine names, strengths, and dates. Confirm every field on the physical package or with a pharmacist. This assistant does not recommend a dose or decide whether a medicine is safe.")
    return results

Run the function below after entering a valid image path and API credentials in the previous cell.

In [ ]:
# OPTIONAL LIVE INFERENCE
medicine_reading_results = read_medicine_bottle(MEDICINE_IMAGE_PATH, confidence_threshold=0.60)

## 7. Medicine assistant output (example format)

The assistant should present **only fields actually found by OCR**, not invent missing information:

- Medicine text: first readable line(s) returned by OCR
- Strength/volume: pattern-matched text such as `500 mg` or `100 ml`
- Batch number: value after `Batch` / `Lot`, if visible
- Manufacturing date: value after `MFG` / `MFD`, if visible
- Expiry date: value after `EXP` / `Expiry`, if visible
- MRP: value after `MRP`, if visible
- Confidence: model detection score plus OCR text scores

**Safety rule:** display “Not detected” when a field is absent or unclear. Ask for a clearer photo rather than guessing. The output is an aid to reading a package, not medical advice.

# PROJECT 2 - Emotion-Aware Cognitive Assistant

## 8. Aim and system design

The assistant receives a face image, detects/crops the face, sends the crop to an image-classification model, and uses the predicted facial-expression category to show a gentle, non-judgmental response. It is a demonstration of expression classification; it must not be presented as a reliable measurement of a person's true feelings or mental health.

**Pipeline:**

`Face image -> face crop -> LandingLens classification endpoint -> expression label + confidence -> optional supportive message`

The uploaded export contains five categories: `neutral`, `happy`, `angry`, `surprise`, and `sad`. The code below keeps those five categories to match the supplied model export.

## 9. Dataset and LandingLens setup

1. Obtain facial-expression images with permission for academic use. One widely used reference dataset is [FER-2013](https://www.kaggle.com/c/challenges-in-representation-learning-facial-expression-recognition-challenge), which contains 48 x 48 grayscale face images. The original FER-2013 dataset has seven categories; the supplied LandingLens export has five. To match your supplied model, use only the five categories present in your export or keep a separate seven-class project and report that difference clearly.
2. Arrange images into folders named `neutral`, `happy`, `angry`, `surprise`, and `sad` (one face per image, with consent and no identity labels).
3. In LandingLens, create a **Classification** project. Upload the images and assign one expression label to each image.
4. Use train/dev/test splits. The CSV supplied here has 400 training images and 100 development images (80/20); it does not contain a separate test split.
5. Train the model. Review class-wise precision, recall, F1, and the confusion matrix, not just overall accuracy.
6. Deploy the model and copy its endpoint ID. Use the same LandingAI API key as above if it belongs to the correct account.

FER-2013 is commonly distributed as 48 x 48 grayscale facial-expression images. The five-class export supplied for this assignment is narrower than the seven-class source dataset; do not silently mix label schemes.

## 10. Load and inspect the supplied emotion result export (Q2 CSV.csv)

This file contains metadata and predictions, not image pixels. It can be used to evaluate the exported predictions and reproduce the confusion matrix, but not to retrain the image classifier by itself.

In [ ]:
Q2_CSV_PATH = Path("Q2 CSV.csv")
if not Q2_CSV_PATH.exists():
    raise FileNotFoundError("Q2 CSV.csv not found. Extract the supplied ZIP or set Q2_CSV_PATH to its location.")

emotion_results = pd.read_csv(Q2_CSV_PATH)
print("Total prediction records:", len(emotion_results))
print("Image sizes:")
print(emotion_results[["Image Width", "Image Height"]].drop_duplicates().to_string(index=False))
print("\\nSplit sizes:")
print(emotion_results["Split"].value_counts())
print("\\nGround-truth class counts:")
print(emotion_results["GT_Class"].value_counts())
display(emotion_results.head())

## 11. Calculate accuracy and confusion matrices

The confusion matrix rows are true/ground-truth classes and columns are predicted classes. A correct prediction appears on the diagonal. The code calculates results separately for Train and Dev, as well as for all 500 rows.

In [ ]:
EMOTION_LABELS = ["neutral", "happy", "angry", "surprise", "sad"]
train_df = emotion_results[emotion_results["Split"].str.lower() == "train"].copy()
dev_df = emotion_results[emotion_results["Split"].str.lower() == "dev"].copy()


def evaluate_split(data, split_name):
    accuracy = accuracy_score(data["GT_Class"], data["PRED_Class"])
    matrix = confusion_matrix(data["GT_Class"], data["PRED_Class"], labels=EMOTION_LABELS)
    print(f"{split_name} images: {len(data)}")
    print(f"{split_name} accuracy: {accuracy:.3%}")
    print(classification_report(
        data["GT_Class"], data["PRED_Class"],
        labels=EMOTION_LABELS, zero_division=0
    ))
    return matrix

train_cm = evaluate_split(train_df, "TRAIN")
dev_cm = evaluate_split(dev_df, "DEV")
all_cm = evaluate_split(emotion_results, "ALL ROWS")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, matrix, title in [
    (axes[0], train_cm, "Train split confusion matrix"),
    (axes[1], dev_cm, "Development split confusion matrix")
]:
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues",
                xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Ground-truth label")
plt.tight_layout()
plt.show()

In [ ]:
accuracy_table = pd.DataFrame({
    "Split": ["Train", "Development", "All exported rows"],
    "Images": [len(train_df), len(dev_df), len(emotion_results)],
    "Accuracy": [
        accuracy_score(train_df["GT_Class"], train_df["PRED_Class"]),
        accuracy_score(dev_df["GT_Class"], dev_df["PRED_Class"]),
        accuracy_score(emotion_results["GT_Class"], emotion_results["PRED_Class"])
    ]
})
accuracy_table["Accuracy (%)"] = (accuracy_table["Accuracy"] * 100).round(1)
display(accuracy_table[["Split", "Images", "Accuracy (%)"]])

### Observed Q2 results from your uploaded CSV

| Evaluation group | Images | Accuracy |
|---|---:|---:|
| Train | 400 | **58.5%** |
| Development | 100 | **41.0%** |
| All exported rows | 500 | **55.0%** |

The supplied confusion-matrix screenshot matches the **400-image Train split** in `Q2 CSV.csv`. Its row totals are 80 per class. The CSV has 100 examples per class across both splits (80 Train + 20 Dev).

**Interpretation:** development accuracy is 41.0%, which is 17.5 percentage points below training accuracy. This gap suggests overfitting or weak generalization. The model performs best on `surprise` and struggles most with `sad` in the supplied predictions. Use the Dev split to improve the model, then evaluate on a separate Test set.

## 12. Optional live inference: face crop + LandingLens classification

This optional section detects the largest face in a photograph using OpenCV's Haar cascade, resizes the crop to 48 x 48 pixels to resemble the supplied image dimensions, and calls your deployed five-class classification endpoint.

For best results, test different lighting, angles, and backgrounds. Do not use facial-expression predictions for recruitment, grading, discipline, medical diagnosis, or other high-impact decisions.

In [ ]:
# OPTIONAL: configure your deployed emotion-classification endpoint.
from landingai.pipeline.image_source import Frame
from landingai.predict import Predictor

EMOTION_ENDPOINT_ID = input("Enter the emotion classification endpoint ID: ").strip()
# LANDINGAI_API_KEY was requested in Project 1. If you skipped Project 1, request it securely:
if "LANDINGAI_API_KEY" not in globals() or not LANDINGAI_API_KEY:
    LANDINGAI_API_KEY = getpass.getpass("Enter your LandingAI API key (input hidden): ")

emotion_predictor = Predictor(
    endpoint_id=EMOTION_ENDPOINT_ID,
    api_key=LANDINGAI_API_KEY
)
print("Emotion endpoint initialized.")

In [ ]:
def crop_largest_face(image_path, output_path="face_crop.jpg", size=(48, 48)):
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    face_detector = cv2.CascadeClassifier(cascade_path)
    faces = face_detector.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
    )

    if len(faces) == 0:
        print("No face detected. Try a clearer, front-facing image with good lighting.")
        return None

    x, y, w, h = max(faces, key=lambda item: item[2] * item[3])
    face = image[y:y+h, x:x+w]
    face = cv2.resize(face, size, interpolation=cv2.INTER_AREA)
    cv2.imwrite(output_path, face)
    return output_path


EMOTION_RESPONSES = {
    "happy": "The visible expression was classified as happy. Keep doing something you enjoy.",
    "sad": "The visible expression was classified as sad. Would a short break or a supportive conversation help?",
    "angry": "The visible expression was classified as angry. Consider pausing and taking a few slow breaths.",
    "surprise": "The visible expression was classified as surprise. Something may have caught your attention.",
    "neutral": "The visible expression was classified as neutral. How are you feeling right now?"
}


def predict_emotion(image_path, confidence_threshold=0.35):
    face_path = crop_largest_face(image_path)
    if face_path is None:
        return None

    frame = Frame.from_image(face_path)
    frame.run_predict(emotion_predictor)
    predictions = list(frame.predictions)
    if not predictions:
        print("The model returned no prediction.")
        return None

    best = max(predictions, key=lambda item: float(item.score))
    label = best.label_name.lower()
    confidence = float(best.score)

    print("Predicted facial-expression label:", label)
    print("Confidence:", round(confidence, 3))

    if confidence < confidence_threshold:
        response = "The model is not confident enough to show a label. Please do not interpret this result."
    else:
        response = EMOTION_RESPONSES.get(label, "No response is configured for this label.")

    print("Assistant response:", response)
    print("Reminder: facial expression is not the same as a person's actual emotional state. Ask the person how they feel.")
    return {"label": label, "confidence": confidence, "response": response}

In [ ]:
# OPTIONAL LIVE INFERENCE
EMOTION_IMAGE_PATH = input("Enter the path to a consented face image: ").strip()
emotion_result = predict_emotion(EMOTION_IMAGE_PATH, confidence_threshold=0.35)

## 13. Design safeguards and limitations

- **Consent and privacy:** obtain permission before using face images; avoid saving images longer than necessary; do not upload private images without authorization.
- **Uncertainty:** facial-expression classifiers can be affected by lighting, pose, occlusion, culture, and individual differences. A predicted label is not proof of internal emotion.
- **Abstention:** use a confidence threshold chosen using the development set. When confidence is low, show an uncertain result rather than forcing a label.
- **Fairness:** evaluate performance across varied lighting, skin tones, ages, camera quality, and other relevant groups where lawful and appropriate.
- **No high-stakes use:** never use this prototype to diagnose mental-health conditions or make consequential decisions about a person.

# 14. Final conclusion

Two LandingLens-based cognitive assistants have been designed:

1. **Medicine-label reading assistant:** a LandingLens object-detection model finds the label region; OCR extracts readable text; simple regular expressions identify possible fields. The supplied model screenshot reports 100% F1/precision/recall on the Train evaluation set. The Q1 CSV contains 87 image records and 91 labelled/predicted instances; the computed average confidence and box IoU are shown in the preceding cells. An independent test set, especially negative images, is required before making real-world performance claims.
2. **Emotion-aware assistant:** a LandingLens classification model assigns one of five expression labels and a cautious, supportive message is displayed. The supplied CSV gives 58.5% Train accuracy, 41.0% Dev accuracy, and 55.0% accuracy across all 500 exported rows. The train/dev gap shows that the model needs further improvement and separate testing.

Both projects follow the cognitive-system cycle: image input, computer-vision inference, extraction/classification, interpretation, and user-facing assistance. Human verification remains essential, particularly for medicine information and facial-expression interpretation.

## References

- GitHub repository supplied with the assignment: https://github.com/VanshikaSohal/COGINITIVE-COMPUTING (contains the Assignment 9 result exports and screenshots used in this analysis).
- LandingAI inference setup: https://landing-ai.github.io/landingai-python/inferences/getting-started/
- LandingAI frame inference: https://landing-ai.github.io/landingai-python/inferences/frames-inference/
- LandingAI OCR: https://landing-ai.github.io/landingai-python/inferences/ocr/
- LandingLens project types: https://support.landing.ai/docs/project-types
- FER-2013 reference dataset: https://www.kaggle.com/c/challenges-in-representation-learning-facial-expression-recognition-challenge
- Medicine-label dataset option: https://universe.roboflow.com/parichat-j/medicine_label-amj7d